# Configuration

In [1]:
import sys
import code
import os
import pickle

import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np

from tqdm import tqdm
import scipy.sparse
from scipy.sparse import csr_matrix
from gtfparse import read_gtf
from collections import defaultdict
from sklearn.metrics import auc, precision_recall_curve, average_precision_score

sys.path.append('/home/wuyan/dygmamba_project/model/dygmamba/src/')

# 1. 加载 autoreload 扩展
%load_ext autoreload

# 2. 设置模式为 "2" (表示自动重载所有模块)
%autoreload 2



In [8]:
data_root= "/home/wuyan/dygmamba_project/"

cell_type = "HELA"


hic_file = data_root + "data/hic/scenhancer/Hela-S3_EP.txt"

data_path = data_root + "data/cell_line/" + cell_type + "/original/"

output_path = data_root + "data/cell_line/" + cell_type + "/process/"

adata_atac_file = output_path + "atac_processed.h5ad"
adata_rna_file = output_path + "rna_processed.h5ad"

adata_rna = ad.read_h5ad(adata_rna_file)
adata_atac = ad.read_h5ad(adata_atac_file)

hic_data_df = pd.read_pickle(output_path + 'hic_data_new.pkl')

gene_info = pd.read_pickle(output_path + "gene_info_filtered.pkl")

hic_data_df['PeakID'] = hic_data_df['chrom'].astype(str) + '-' + \
hic_data_df['region_start'].astype(str) + '-' + hic_data_df['region_end'].astype(str)
hic_data_df.rename(columns={'chrom':'chr', 'region_start':'start', 
                            'region_end':'end', 'gene_name':'gene',
                            'abc_score':'hic_score'}, inplace= True)

hic_data_df = hic_data_df[hic_data_df['gene'].isin(set(adata_rna.var_names))]

from pdata.data_preprocess import build_hic_peak_gene_network_schic

peak_gene_df, peak_gene_grn = build_hic_peak_gene_network_schic(adata_atac, hic_data_df, score_col="hic_score") 



--- 1. 数据准备 (Converting to PyRanges) ---
  -> 使用完整 peak 区间与 Hi-C 区间 overlap
  -> Hi-C region-gene 区间数: 220756
--- 2. 寻找 Peak-Gene 连接 (Overlap peak with Hi-C region-gene) ---
共找到 62391 条唯一的 Peak-Gene 连接。
--- 3. 构建稀疏矩阵 ---
✅ 构建完成! 矩阵维度: (63211, 9516) (Peaks x Genes)


,chr,start,end,gene,ensembl_id,tss,hic_score,PeakID
0,chr1,880180,880430,AL669831.1,ENSG00000197049,721320,0.839241,chr1-880180-880430
1,chr1,902000,902080,AL669831.1,ENSG00000197049,721320,0.820137,chr1-902000-902080
2,chr1,934990,935070,AL669831.1,ENSG00000197049,721320,0.622639,chr1-934990-935070
3,chr1,955750,957580,AL669831.1,ENSG00000197049,721320,1.165843,chr1-955750-957580
4,chr1,1019880,1021250,AL669831.1,ENSG00000197049,721320,2.176111,chr1-1019880-1021250
...,...,...,...,...,...,...,...,...
220751,chrX,154123680,154124350,MTCP1NB,ENSG00000182712,154299637,0.812218,chrX-154123680-154124350
220752,chrX,153927790,153928270,BRCC3,ENSG00000185515,154299695,0.928908,chrX-153927790-153928270
220753,chrX,154293680,154293760,BRCC3,ENSG00000185515,154299695,1.977747,chrX-154293680-154293760
220754,chrX,154293810,154293890,BRCC3,ENSG00000185515,154299695,1.967495,chrX-154293810-154293890


In [9]:
hic_data_df = hic_data_df[hic_data_df['gene'].isin(set(adata_rna.var_names))]

hic_data_df


,chr,start,end,gene,ensembl_id,tss,hic_score,PeakID
96,chr1,877870,877930,MTND2P28,ENSG00000225630,565020,1.470994,chr1-877870-877930
97,chr1,878440,878630,MTND2P28,ENSG00000225630,565020,2.052369,chr1-878440-878630
98,chr1,878760,879070,MTND2P28,ENSG00000225630,565020,2.052369,chr1-878760-879070
99,chr1,879190,879280,MTND2P28,ENSG00000225630,565020,1.310022,chr1-879190-879280
100,chr1,880180,880430,MTND2P28,ENSG00000225630,565020,1.479415,chr1-880180-880430
...,...,...,...,...,...,...,...,...
220653,chrX,154019340,154020040,DKC1,ENSG00000130826,153991031,2.430974,chrX-154019340-154020040
220654,chrX,154020120,154020410,DKC1,ENSG00000130826,153991031,2.354586,chrX-154020120-154020410
220655,chrX,153714670,153714870,DKC1,ENSG00000130826,153991031,2.084191,chrX-153714670-153714870
220744,chrX,154293680,154293760,FUNDC2,ENSG00000165775,154254255,0.947769,chrX-154293680-154293760


In [ ]:
hic_data_df['PeakID'] = hic_data_df['chrom'].astype(str) + '-' + \
hic_data_df['region_start'].astype(str) + '-' + hic_data_df['region_end'].astype(str)

peak_gene_df = hic_data_df[['PeakID', 'gene_name', 'abc_score']]

peak_gene_df.rename(columns={'abc_score':'hic_score'}, inplace= True)


all_peaks = atac_adata.var_names
peak_to_idx = {peak: i for i, peak in enumerate(all_peaks)}

all_genes = sorted(peak_gene_df['gene_name'].unique())
gene_to_idx = {gene: i for i, gene in enumerate(all_genes)}

rows = peak_gene_df['PeakID'].map(peak_to_idx).values
cols = peak_gene_df['gene_name'].map(gene_to_idx).values
data = peak_gene_df["hic_score"]
sparse_matrix = sparse.csr_matrix(
        (data, (rows, cols)), 
        shape=(len(all_peaks), len(all_genes))
    )
sparse_matrix.data = np.where(sparse_matrix.data > 0, 1, 0)

peak_gene_grn = ad.AnnData(
    X=sparse_matrix,
    obs=pd.DataFrame(index=all_peaks), # 行索引严格对齐 ATAC Peak
    var=pd.DataFrame(index=all_genes)  # 列索引是 Gene
)

peak_gene_grn.uns['description'] = 'Peak-Gene Binary Connectivity Matrix'

/tmp/ipykernel_52604/2583618492.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  peak_gene_df.rename(columns={'abc_score':'hic_score'}, inplace= True)


,PeakID,gene_name,hic_score
0,chr1-880180-880430,AL669831.1,0.839241
1,chr1-902000-902080,AL669831.1,0.820137
2,chr1-934990-935070,AL669831.1,0.622639
3,chr1-955750-957580,AL669831.1,1.165843
4,chr1-1019880-1021250,AL669831.1,2.176111
...,...,...,...
220751,chrX-154123680-154124350,MTCP1NB,0.812218
220752,chrX-153927790-153928270,BRCC3,0.928908
220753,chrX-154293680-154293760,BRCC3,1.977747
220754,chrX-154293810-154293890,BRCC3,1.967495


# Function

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sparse
import anndata as ad
import pyranges as pr


def build_hic_peak_gene_network_schic(
    atac_adata,
    hic_df,
    score_col=None,
    use_peak_center=False,
    binary=True,
    agg_method="max"
):
    """
    根据 Hi-C region-gene 表（chr, start, end, gene）构建 Peak-Gene 网络。

    规则：
    只要 ATAC peak 与 Hi-C 区间 [chr, start, end] 有重叠，
    就认为该 peak 与对应 gene 存在一条边。

    Parameters
    ----------
    atac_adata : AnnData
        ATAC-seq 数据对象。要求 var_names 能唯一标识 peak，
        且 get_atac_bed()/get_atac_peak_centers() 返回结果中包含 'PeakID' 列。

    hic_df : pd.DataFrame
        Hi-C region-gene 数据表，至少包含以下列：
        - chr
        - start
        - end
        - gene
        可选包含一个分数字段 score_col。

    gene_info_df : pd.DataFrame or None, optional
        为了兼容旧接口而保留，但当前版本不使用。

    score_col : str or None, optional
        Hi-C 边权重列名。
        - 若为 None，则每条 overlap 边权重记为 1
        - 若提供，则使用该列作为边权重
        当同一 Peak-Gene 对应多个区间时，会按 agg_method 聚合。

    use_peak_center : bool, default=False
        - False: 用完整 peak 区间与 Hi-C 区间 overlap
        - True : 用 peak 中心点与 Hi-C 区间 overlap（更严格）

    binary : bool, default=True
        - True : 输出二值矩阵（有边=1）
        - False: 输出加权矩阵（权重来自 score_col 或默认 1）

    agg_method : {"max", "sum", "mean"}, default="max"
        当同一 Peak-Gene 由于多个 Hi-C 区间重复出现时，如何聚合边权重。

    Returns
    -------
    peak_gene_df : pd.DataFrame
        包含三列：
        - PeakID
        - gene_name
        - hic_score

    peak_gene_grn : AnnData
        Peaks x Genes 的连接矩阵：
        - obs.index = all peaks
        - var.index = connected genes

    Notes
    -----
    1. 本函数适用于 Hi-C 已整理成 region-gene 配对表的情况。
    2. 这构建的是 peak-gene 候选连接网络 / prior network，
       不是原始 loop 双端严格重建。
    """

    print("--- 1. 数据准备 (Converting to PyRanges) ---")

    # ------------------------------------------------------------------
    # 1) 获取 peak 区间
    # ------------------------------------------------------------------
    if use_peak_center:
        pr_peaks = get_atac_peak_centers(atac_adata)
        print("  -> 使用 peak center 与 Hi-C 区间 overlap")
    else:
        pr_peaks = get_atac_bed(atac_adata)
        print("  -> 使用完整 peak 区间与 Hi-C 区间 overlap")

    if pr_peaks is None:
        print("❌ 错误: 无法从 atac_adata 提取 peak 区间信息。")
        return None

    peak_df = pr_peaks.df.copy()
    if 'PeakID' not in peak_df.columns:
        raise ValueError(
            "get_atac_bed() / get_atac_peak_centers() 返回的 PyRanges 中必须包含 'PeakID' 列。"
        )

    # ------------------------------------------------------------------
    # 2) 检查并标准化 Hi-C region-gene 表
    # ------------------------------------------------------------------
    if hic_df is None or len(hic_df) == 0:
        print("❌ 错误: hic_df 为空。")
        return None

    hic_df = hic_df.copy()

    required_cols = ['chr', 'start', 'end', 'gene']
    missing_cols = [c for c in required_cols if c not in hic_df.columns]
    if missing_cols:
        raise ValueError(f"hic_df 缺少必要列: {missing_cols}")

    # 基础清洗
    hic_df = hic_df.dropna(subset=['chr', 'start', 'end', 'gene']).copy()
    hic_df['start'] = hic_df['start'].astype(int)
    hic_df['end'] = hic_df['end'].astype(int)
    hic_df['gene'] = hic_df['gene'].astype(str)

    # 去除非法区间
    hic_df = hic_df[hic_df['end'] > hic_df['start']].copy()
    if hic_df.empty:
        print("❌ 错误: hic_df 清洗后没有合法区间。")
        return None

    # 处理 score
    if score_col is not None:
        if score_col not in hic_df.columns:
            raise ValueError(f"score_col='{score_col}' 不在 hic_df 中。")
        hic_df[score_col] = pd.to_numeric(hic_df[score_col], errors='coerce')
        hic_df[score_col] = hic_df[score_col].fillna(0.0)
        used_score_col = score_col
    else:
        hic_df['hic_score'] = 1.0
        used_score_col = 'hic_score'

    # 转为 PyRanges 格式
    hic_pr_df = hic_df.rename(columns={
        'chr': 'Chromosome',
        'start': 'Start',
        'end': 'End',
        'gene': 'gene_name'
    })[['Chromosome', 'Start', 'End', 'gene_name', used_score_col]].copy()

    pr_hic = pr.PyRanges(hic_pr_df)

    print(f"  -> Hi-C region-gene 区间数: {len(hic_pr_df)}")

    # ------------------------------------------------------------------
    # 3) overlap：peak × Hi-C region-gene
    # ------------------------------------------------------------------
    print("--- 2. 寻找 Peak-Gene 连接 (Overlap peak with Hi-C region-gene) ---")

    overlap_df = pr_peaks.join(pr_hic).df

    if overlap_df.empty:
        print("❌ 警告: 未找到任何 Peak-Gene 连接！")
        return None

    required_overlap_cols = ['PeakID', 'gene_name', used_score_col]
    missing_overlap_cols = [c for c in required_overlap_cols if c not in overlap_df.columns]
    if missing_overlap_cols:
        raise ValueError(
            f"overlap 结果中缺少必要列: {missing_overlap_cols}\n"
            f"请检查 get_atac_bed()/get_atac_peak_centers() 是否保留了 PeakID。"
        )

    peak_gene_df = overlap_df[['PeakID', 'gene_name', used_score_col]].copy()
    peak_gene_df = peak_gene_df.rename(columns={used_score_col: 'hic_score'})

    # ------------------------------------------------------------------
    # 4) 聚合同一 Peak-Gene 的重复边
    # ------------------------------------------------------------------
    if agg_method == "max":
        peak_gene_df = peak_gene_df.groupby(
            ['PeakID', 'gene_name'], as_index=False
        )['hic_score'].max()
    elif agg_method == "sum":
        peak_gene_df = peak_gene_df.groupby(
            ['PeakID', 'gene_name'], as_index=False
        )['hic_score'].sum()
    elif agg_method == "mean":
        peak_gene_df = peak_gene_df.groupby(
            ['PeakID', 'gene_name'], as_index=False
        )['hic_score'].mean()
    else:
        raise ValueError("agg_method 只能是 'max', 'sum', 'mean'")

    if peak_gene_df.empty:
        print("❌ 警告: 聚合后没有任何 Peak-Gene 连接！")
        return None

    print(f"共找到 {len(peak_gene_df)} 条唯一的 Peak-Gene 连接。")

    # ------------------------------------------------------------------
    # 5) 构建稀疏矩阵
    # ------------------------------------------------------------------
    print("--- 3. 构建稀疏矩阵 ---")

    all_peaks = atac_adata.var_names.tolist()
    peak_gene_df = peak_gene_df[peak_gene_df['PeakID'].isin(all_peaks)].copy()

    if peak_gene_df.empty:
        print("❌ 警告: overlap 得到的 PeakID 与 atac_adata.var_names 没有交集。")
        return None

    all_genes = sorted(peak_gene_df['gene_name'].unique())

    peak_to_idx = {peak: i for i, peak in enumerate(all_peaks)}
    gene_to_idx = {gene: i for i, gene in enumerate(all_genes)}

    rows = peak_gene_df['PeakID'].map(peak_to_idx).values
    cols = peak_gene_df['gene_name'].map(gene_to_idx).values
    data = peak_gene_df['hic_score'].astype(float).values

    sparse_matrix = sparse.csr_matrix(
        (data, (rows, cols)),
        shape=(len(all_peaks), len(all_genes))
    )

    if binary:
        sparse_matrix.data = np.where(sparse_matrix.data > 0, 1, 0)

    peak_gene_grn = ad.AnnData(
        X=sparse_matrix,
        obs=pd.DataFrame(index=all_peaks),
        var=pd.DataFrame(index=all_genes)
    )

    peak_gene_grn.uns['description'] = 'Peak-Gene connectivity matrix built from Hi-C region-gene overlap'
    peak_gene_grn.uns['source'] = 'Hi-C region-gene table'
    peak_gene_grn.uns['use_peak_center'] = use_peak_center
    peak_gene_grn.uns['binary'] = binary
    peak_gene_grn.uns['agg_method'] = agg_method
    peak_gene_grn.uns['score_col'] = None if score_col is None else score_col

    print(f"✅ 构建完成! 矩阵维度: {peak_gene_grn.shape} (Peaks x Genes)")

    return peak_gene_df, peak_gene_grn